 # Evolução dos Casos de Dengue no Brasil (2015-2024)
## Projeto G2 — Linguagem de Programação

---

**Alunos:** Ariene Viana e João Marcelo Lopes

**Periodo analisado:** 2015 a 2024

---

## 1. Introdução

A dengue e uma das doenças que mais preocupam o Brasil em termos de saúde publica.
Transmitida pelo mosquito *Aedes aegypti*, ela afeta milhares de pessoas todos os anos,
especialmente nos períodos de verão, quando as chuvas criam condições ideais para a
reprodução do mosquito.

Neste projeto, vamos analisar os dados de dengue no Brasil entre 2015 e 2024,
buscando entender melhor como a doença se comporta ao longo do tempo e do território.

**O que queremos descobrir:**
- Quais estados tem mais casos de dengue?
- Em quais meses do ano os casos aumentam mais?
- Os casos cresceram ao longo dos anos?
- Existe relação entre chuva e aumento de casos?
- Quais municípios são mais afetados?

In [ ]:
!pip install plotly sqlalchemy --quiet

import warnings
warnings.filterwarnings('ignore')


import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


from sqlalchemy import create_engine, text


sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13,
                     'axes.labelsize': 11})


CORES = {'Norte':'#3182ce','Nordeste':'#dd6b20',
         'Centro-Oeste':'#38a169','Sudeste':'#e53e3e','Sul':'#805ad5'}


MESES = ['Jan','Fev','Mar','Abr','Mai','Jun',
         'Jul','Ago','Set','Out','Nov','Dez']

print('Bibliotecas carregadas com sucesso!')

Bibliotecas carregadas com sucesso!


## 2. Upload e Leitura do CSV

In [ ]:
from google.colab import files

print('Clique em Escolher arquivos e selecione:')
print('   simulacao_dengue_brasil.csv')
print()

uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]


df_raw = pd.read_csv(nome_arquivo)
print(f'Arquivo carregado: {nome_arquivo}')
print(f'{df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas')
df_raw.head()

Clique em Escolher arquivos e selecione:
   simulacao_dengue_brasil.csv



Saving simulacao_dengue_brasil.csv to simulacao_dengue_brasil.csv
Arquivo carregado: simulacao_dengue_brasil.csv
4,440 linhas x 14 colunas


,ano,mes,data,regiao,uf,municipio,populacao,chuva_mm,temperatura_media,casos_dengue,internacoes,obitos,incidencia_100k,nivel_alerta
0,2015,1,2015-01-01,Norte,AM,Manaus,2708209,141.6,26.6,998,43,2,36.85,Médio
1,2015,1,2015-01-01,Norte,PA,Belém,711775,161.5,26.4,229,12,0,32.17,Médio
2,2015,1,2015-01-01,Norte,PA,Santarém,2249561,128.9,26.1,776,35,2,34.50,Médio
3,2015,1,2015-01-01,Norte,RO,Porto Velho,2198291,180.8,28.5,915,51,3,41.62,Médio
4,2015,1,2015-01-01,Norte,TO,Palmas,2193271,151.1,24.7,760,37,0,34.65,Médio


## 3. Limpeza e Preparação dos Dados

In [ ]:
print('=== INFORMACOES DO DATASET ===')
df_raw.info()
print('\n=== VALORES NULOS ===')
print(df_raw.isnull().sum())
print('\n=== ESTATISTICAS DESCRITIVAS ===')
display(df_raw.describe().round(2))

=== INFORMACOES DO DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4440 entries, 0 to 4439
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ano                4440 non-null   int64  
 1   mes                4440 non-null   int64  
 2   data               4440 non-null   object 
 3   regiao             4440 non-null   object 
 4   uf                 4440 non-null   object 
 5   municipio          4440 non-null   object 
 6   populacao          4440 non-null   int64  
 7   chuva_mm           4440 non-null   float64
 8   temperatura_media  4440 non-null   float64
 9   casos_dengue       4440 non-null   int64  
 10  internacoes        4440 non-null   int64  
 11  obitos             4440 non-null   int64  
 12  incidencia_100k    4440 non-null   float64
 13  nivel_alerta       4440 non-null   object 
dtypes: float64(3), int64(6), object(5)
memory usage: 485.8+ KB

=== VALORES NULOS ===
ano    

,ano,mes,populacao,chuva_mm,temperatura_media,casos_dengue,internacoes,obitos,incidencia_100k
count,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00
mean,2019.50,6.50,2589413.97,98.89,26.02,801.93,40.24,1.20,30.94
std,2.87,3.45,1392966.82,49.21,2.96,465.92,24.20,1.32,5.94
min,2015.00,1.00,150048.00,0.00,15.30,26.00,0.00,0.00,11.69
25%,2017.00,3.75,1404289.75,62.78,24.60,413.00,20.00,0.00,26.83
50%,2019.50,6.50,2608171.50,94.60,26.50,774.50,38.00,1.00,30.86
75%,2022.00,9.25,3781211.25,135.30,28.00,1148.25,58.00,2.00,35.06
max,2024.00,12.00,4994111.00,247.60,33.80,2239.00,119.00,8.00,51.97


In [ ]:
df = df_raw.copy()


df['data'] = pd.to_datetime(df['data'])


for c in ['ano','mes','populacao','casos_dengue','internacoes','obitos']:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)
for c in ['chuva_mm','temperatura_media','incidencia_100k']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['regiao']       = df['regiao'].str.strip().str.title()
df['uf']           = df['uf'].str.strip().str.upper()
df['municipio']    = df['municipio'].str.strip().str.title()
df['nivel_alerta'] = df['nivel_alerta'].str.strip().str.title()


antes = len(df)
df = df.drop_duplicates()
print(f'Duplicatas removidas: {antes - len(df)}')
print(f'Dataset limpo: {df.shape[0]:,} linhas | Nulos restantes: {df.isnull().sum().sum()}')

## 4. Engenharia de Atríbutos (Novas Colunas)

In [ ]:
# Vou criar algumas colunas novas que vao ajudar nas analises

# Nome do mes em portugues para deixar os graficos mais legiveis
df['nome_mes'] = df['mes'].apply(lambda x: MESES[x-1])

# Estacao do ano considerando o hemisferio Sul
def estacao(m):
    if m in [12,1,2]:  return 'Verao'
    if m in [3,4,5]:   return 'Outono'
    if m in [6,7,8]:   return 'Inverno'
    return 'Primavera'
df['estacao'] = df['mes'].apply(estacao)

# Taxa de letalidade: quantos casos resultaram em obito
df['taxa_letalidade'] = np.where(df['casos_dengue']>0,
    df['obitos']/df['casos_dengue']*100, 0).round(4)

# Taxa de internacao: quantos casos resultaram em internacao
df['taxa_internacao'] = np.where(df['casos_dengue']>0,
    df['internacoes']/df['casos_dengue']*100, 0).round(2)

print('Novas colunas criadas com sucesso!')
df[['nome_mes','estacao','taxa_letalidade','taxa_internacao']].head(3)

## 5. KPIs — Indicadores Principais

In [ ]:
# Calculando os principais indicadores do periodo analisado
total_casos  = df['casos_dengue'].sum()
total_ob     = df['obitos'].sum()
total_int    = df['internacoes'].sum()
media_men    = df.groupby(['ano','mes'])['casos_dengue'].sum().mean()
inc_media    = df['incidencia_100k'].mean()
uf_top       = df.groupby('uf')['casos_dengue'].sum().idxmax()
uf_top_v     = df.groupby('uf')['casos_dengue'].sum().max()
mun_top      = df.groupby('municipio')['incidencia_100k'].mean().idxmax()
mun_inc      = df.groupby('municipio')['incidencia_100k'].mean().max()
ano_pico     = df.groupby('ano')['casos_dengue'].sum().idxmax()
ano_pico_v   = df.groupby('ano')['casos_dengue'].sum().max()

print('=' * 55)
print('      PAINEL DE KPIs - DENGUE NO BRASIL')
print('=' * 55)
print(f'  Total de Casos          : {total_casos:>12,.0f}')
print(f'  Total de Obitos         : {total_ob:>12,.0f}')
print(f'  Total de Internacoes    : {total_int:>12,.0f}')
print(f'  Media Mensal de Casos   : {media_men:>12,.0f}')
print(f'  Incidencia Media /100k  : {inc_media:>12.1f}')
print(f'  Estado Mais Afetado     : {uf_top} ({uf_top_v:,.0f} casos)')
print(f'  Municipio Critico       : {mun_top} ({mun_inc:.1f}/100k)')
print(f'  Ano de Pico             : {ano_pico} ({ano_pico_v:,.0f} casos)')
print('=' * 55)

## 6. Gráficos

### 6.1 Evolução Temporal dos Casos

In [ ]:
# GRAFICO 1 - Evolucao anual dos casos de dengue
#
# Escolhi esse grafico de linha porque ele e o melhor para mostrar
# como uma coisa muda ao longo do tempo. Da pra ver claramente
# em quais anos a situacao piorou ou melhorou.
# A area sombreada embaixo da linha ajuda a ter uma nocao do volume total.
# As barras do lado mostram obitos e internacoes para complementar a analise.

por_ano = df.groupby('ano').agg(
    casos=('casos_dengue','sum'),
    obitos=('obitos','sum'),
    internacoes=('internacoes','sum')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Grafico de linha com area sombreada
axes[0].plot(por_ano['ano'], por_ano['casos'],
             marker='o', lw=2.5, color='#e53e3e', ms=8)
axes[0].fill_between(por_ano['ano'], por_ano['casos'],
                      alpha=0.15, color='#e53e3e')
axes[0].set_title('Evolucao Anual dos Casos')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Total de Casos')
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
for x,y in zip(por_ano['ano'], por_ano['casos']):
    axes[0].annotate(f'{y:,.0f}', (x,y),
                     xytext=(0,8), textcoords='offset points',
                     ha='center', fontsize=8)

# Barras de obitos e internacoes para comparacao
x = np.arange(len(por_ano)); w = 0.38
axes[1].bar(x-w/2, por_ano['obitos'],     w,
            label='Obitos',      color='#8c1a11')
axes[1].bar(x+w/2, por_ano['internacoes'],w,
            label='Internacoes', color='#ed8936')
axes[1].set_title('Obitos e Internacoes por Ano')
axes[1].set_xticks(x)
axes[1].set_xticklabels(por_ano['ano'], rotation=45)
axes[1].legend()

plt.suptitle('Serie Historica - Dengue no Brasil (2015-2024)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Analise:** Observando o gráfico conseguimos identificar os anos com maior
número de casos. Picos epidemicos costumam ocorrer quando um novo sorotipo
do vírus se espalha pela população ou quando as condições climáticas são
especialmente favoraveis para o mosquito. Vale notar se ha uma tendência
de crescimento ou redução ao longo da série histórica.

### 6.2 Comparação entre Regiões

In [ ]:
# GRAFICO 2 - Comparacao de casos entre as regioes do Brasil
#
# Usei barras horizontais junto com um grafico de pizza para mostrar
# tanto o valor absoluto quanto a proporcao de cada regiao.
# Isso ajuda a responder: qual regiao e mais afetada e qual e o peso
# de cada uma no total nacional?
# As cores foram escolhidas para diferenciar bem cada regiao.

por_reg = df.groupby('regiao')['casos_dengue'].sum()\
            .sort_values(ascending=False).reset_index()
por_reg['pct'] = (por_reg['casos_dengue']/por_reg['casos_dengue'].sum()*100).round(1)
cores_lista = [CORES.get(r,'#aaa') for r in por_reg['regiao']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras horizontais com os valores e percentuais
axes[0].barh(por_reg['regiao'], por_reg['casos_dengue'], color=cores_lista)
axes[0].set_title('Total de Casos por Regiao')
axes[0].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
for i,(v,p) in enumerate(zip(por_reg['casos_dengue'], por_reg['pct'])):
    axes[0].text(v+100, i, f' {v:,.0f} ({p}%)', va='center', fontsize=9)

# Pizza mostrando a participacao percentual de cada regiao
axes[1].pie(por_reg['casos_dengue'], labels=por_reg['regiao'],
            autopct='%1.1f%%', colors=cores_lista, startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Distribuicao % por Regiao')

plt.suptitle('Casos de Dengue por Regiao', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Análise:** A distribuição regional mostra que a dengue nao afeta o Brasil
de forma uniforme. Regiões com clima mais quente e úmido tendem a concentrar
mais casos, pois o *Aedes aegypti* se desenvolve melhor nessas condições.
O gráfico de pizza deixa claro o peso relativo de cada região no total nacional.

In [ ]:
# GRAFICO 3 - Evolucao por regiao ao longo do tempo (interativo)
#
# Aqui usei o Plotly para fazer um grafico interativo, onde da pra passar
# o mouse em cima dos pontos e ver os valores exatos.
# O objetivo e comparar como cada regiao evoluiu ao longo dos anos,
# verificando se todas cresceram juntas ou se algumas foram mais afetadas
# em determinados periodos.

por_reg_ano = df.groupby(['ano','regiao'])['casos_dengue'].sum().reset_index()
fig = px.line(por_reg_ano, x='ano', y='casos_dengue', color='regiao',
              markers=True, color_discrete_map=CORES,
              title='Evolucao por Regiao (2015-2024)',
              labels={'casos_dengue':'Casos','ano':'Ano','regiao':'Regiao'})
fig.update_layout(hovermode='x unified')
fig.show()

**Analise:** O gráfico interativo permite comparar o ritmo de crescimento
de cada região. É possivel identificar se os anos de pico foram sincronizados
em todo o país ou se afetaram regiões diferentes em momentos distintos.
Passe o mouse sobre os pontos para ver os valores exatos de cada ano.

### 6.3 Comparação entre Estados (UF)

In [ ]:
# GRAFICO 4 - Total de casos por estado
#
# Esse grafico de barras mostra todos os estados ao mesmo tempo,
# coloridos pela regiao a que pertencem.
# Escolhi esse formato para facilitar a comparacao entre estados
# e tambem para ver quais regioes concentram os estados mais afetados.
# Quanto mais alta a barra, maior o numero total de casos no periodo.

por_uf = df.groupby(['uf','regiao'])['casos_dengue'].sum()\
           .reset_index().sort_values('casos_dengue', ascending=False)
cores_uf = [CORES.get(r,'#aaa') for r in por_uf['regiao']]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(por_uf['uf'], por_uf['casos_dengue'], color=cores_uf, edgecolor='white')
ax.set_title('Casos de Dengue por Estado (UF)', fontsize=13, fontweight='bold')
ax.set_xlabel('Estado')
ax.set_ylabel('Total de Casos')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=v,label=k) for k,v in CORES.items()],
          title='Regiao', loc='upper right')
plt.tight_layout()
plt.show()

**Analise:** O ranking de estados mostra quais unidades federativas concentram
a maior carga da dengue no Brasil. Estados com grandes centros urbanos e clima
favorável tendem a liderar o ranking. É importante comparar tanto o volume
absoluto de casos quanto a incidência por habitante para uma análise mais justa.

### 6.4 Ranking de Municípios

In [ ]:
# GRAFICO 5 - Municipios com maior incidencia media
#
# Nesse grafico decidi usar a incidencia (casos por 100 mil habitantes)
# em vez do total de casos, porque isso e mais justo.
# Por exemplo, uma cidade pequena com 500 casos pode ser mais grave
# do que uma cidade grande com 2000 casos, dependendo do tamanho
# da populacao. As barras horizontais facilitam a leitura dos nomes.

rank = df.groupby(['municipio','uf','regiao'])\
         .agg(casos=('casos_dengue','sum'),
              incidencia=('incidencia_100k','mean'),
              obitos=('obitos','sum'))\
         .reset_index()\
         .sort_values('incidencia', ascending=False)\
         .head(15)

cores_m = [CORES.get(r,'#aaa') for r in rank['regiao']]
labels  = rank['municipio'] + '/' + rank['uf']

fig, ax = plt.subplots(figsize=(13, 7))
ax.barh(labels[::-1], rank['incidencia'][::-1], color=cores_m[::-1])
ax.set_title('Top 15 Municipios - Incidencia Media por 100k hab.',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Incidencia Media / 100 mil habitantes')
for i,v in enumerate(rank['incidencia'][::-1]):
    ax.text(v+0.3, i, f'{v:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\nTabela - Top 15 Municipios')
display(rank.reset_index(drop=True).style.format(
    {'casos':'{:,.0f}','incidencia':'{:.1f}','obitos':'{:,.0f}'}
).background_gradient(subset=['incidencia'], cmap='OrRd'))

**Analise:** O ranking de municípios pela incidência média revela os locais
com maior concentração proporcional de casos. Esses municípios merecem
atenção prioritária das autoridades sanitárias, pois indicam áreas onde
o controle do mosquito é mais urgente. A tabela abaixo do gráfico traz
os números detalhados de cada município.

### 6.5 Heatmap de Sazonalidade

In [ ]:
# GRAFICO 6 - Heatmap de sazonalidade (mes x ano)
#
# O heatmap e perfeito para esse tipo de analise porque permite ver
# ao mesmo tempo o comportamento mensal e anual dos casos.
# Cada celula representa um mes de um determinado ano.
# Quanto mais escura a cor, mais casos naquele periodo.
# Esse grafico responde a pergunta: existe um padrao que se repete todo ano?

pivot = df.groupby(['ano','mes'])['casos_dengue'].sum()\
          .unstack('mes').fillna(0)
pivot.columns = [MESES[c-1] for c in pivot.columns]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='YlOrRd',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label':'Casos'}, ax=ax)
ax.set_title('Heatmap de Sazonalidade - Casos por Mes e Ano',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Ano')
plt.tight_layout()
plt.show()

**Analise:** O heatmap deixa claro o padrão sazonal da dengue. As células
mais escuras se concentram nos primeiros meses do ano (janeiro a abril),
que correspondem ao verão no hemisfério Sul. Esse é o período de maior
calor e chuvas, favorecendo a proliferação do mosquito. Esse padrão
se repete praticamente em todos os anos analisados.

In [ ]:
# GRAFICO 7 - Media historica mensal
#
# Depois de ver o heatmap, quis resumir o padrao sazonal em um unico grafico.
# Calculei a media de casos para cada mes considerando todos os anos.
# A barra vermelha destaca o mes que historicamente tem mais casos.
# Isso ajuda a planejar quando as campanhas de prevencao devem ser intensificadas.

sazon = df.groupby('mes')['casos_dengue'].mean().reset_index()
sazon['nome_mes'] = sazon['mes'].apply(lambda x: MESES[x-1])
mes_pico = sazon.loc[sazon['casos_dengue'].idxmax(), 'nome_mes']

cores_b = ['#e53e3e' if v==sazon['casos_dengue'].max()
           else '#aec7e8' for v in sazon['casos_dengue']]
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(sazon['nome_mes'], sazon['casos_dengue'],
       color=cores_b, edgecolor='white')
ax.set_title('Media Historica Mensal de Casos', fontsize=13, fontweight='bold')
ax.set_ylabel('Media de Casos')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
for i,v in enumerate(sazon['casos_dengue']):
    ax.text(i, v+5, f'{v:,.0f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

print(f'Mes historicamente mais critico: {mes_pico}')

**Analise:** O gráfico de média mensal confirma o padrão identificado no heatmap.
O mês em vermelho e o mais crítico historicamente. Esse tipo de informação
e essencial para o planejamento de ações preventivas, os agentes de saúde
devem intensificar o trabalho de campo nos meses anteriores ao pico,
preferencialmente entre outubro e dezembro.

### 6.6 Relação entre Chuva e Casos de Dengue

In [ ]:
# GRAFICO 8 - Relacao entre chuva e casos ao longo dos meses
#
# Aqui quero mostrar se os meses com mais chuva tambem tem mais casos.
# Usei um grafico com eixo duplo: as barras representam a chuva media
# e a linha representa os casos medios. Assim da pra comparar os dois
# ao mesmo tempo e ver se existe uma relacao visual entre eles.
# A teoria e que mais chuva = mais agua parada = mais mosquito = mais dengue.

med = df.groupby('mes').agg(
    chuva=('chuva_mm','mean'),
    casos=('casos_dengue','mean')).reset_index()
med['nome_mes'] = med['mes'].apply(lambda x: MESES[x-1])

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.bar(med['nome_mes'], med['chuva'],
        color='#4299e1', alpha=0.6, label='Chuva media (mm)')
ax2.plot(med['nome_mes'], med['casos'],
         color='#e53e3e', marker='o', lw=2.5, label='Casos medios')

ax1.set_title('Chuva e Casos de Dengue - Medias Mensais Historicas',
              fontsize=13, fontweight='bold')
ax1.set_xlabel('Mes')
ax1.set_ylabel('Chuva media (mm)', color='#4299e1')
ax2.set_ylabel('Casos medios', color='#e53e3e')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
plt.tight_layout()
plt.show()

# Calculando a correlacao de Pearson
r = df[['chuva_mm','casos_dengue']].corr().iloc[0,1]
forca = 'FORTE' if abs(r)>.6 else 'MODERADA' if abs(r)>.3 else 'FRACA'
print(f'Correlacao de Pearson (chuva x casos): r = {r:.4f} -> {forca}')

**Analise:** O gráfico com eixo duplo permite observar visualmente se os
picos de chuva coincidem com os picos de casos. Note que existe uma
defasagem de aproximadamente 2 a 4 semanas entre o aumento das chuvas
e o aumento dos casos, esse é o tempo necessário para o ciclo de vida
do mosquito se completar. O coeficiente de Pearson confirma numericamente
a força dessa relação.

In [ ]:
# GRAFICO 9 - Matriz de correlacao entre todas as variaveis numericas
#
# Esse tipo de grafico mostra como cada variavel se relaciona com as outras.
# Os valores vao de -1 a 1: quanto mais proximo de 1, mais as duas variaveis
# crescem juntas. Quanto mais proximo de -1, quando uma sobe a outra desce.
# Usei apenas metade da matriz (triangulo inferior) para evitar repeticao.
# E um grafico muito usado em projetos de ciencia de dados.

cols = ['chuva_mm','temperatura_media','casos_dengue',
        'internacoes','obitos','incidencia_100k']
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(df[cols].corr(), dtype=bool))
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f',
            cmap='RdBu_r', vmin=-1, vmax=1,
            mask=mask, linewidths=0.5, square=True, ax=ax)
ax.set_title('Matriz de Correlacao entre Variaveis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

**Analise:** A matriz de correlação revela quais variaveis estão mais
associadas entre si. Espera-se encontrar correlação positiva entre casos,
internações e óbitos (variáveis epidemiológicas que tendem a crescer juntas),
e também alguma correlação entre chuva e casos. Variavéis com correlação
próxima de zero não tem relação linear significativa.

### 6.7 Tabela Dinâmica — Casos por UF e Ano

In [ ]:
# TABELA DINAMICA - Casos por estado e por ano
#
# Alem dos graficos, achei importante mostrar os dados em formato de tabela
# para quem quiser consultar os numeros exatos de cada estado em cada ano.
# O gradiente de cores em vermelho facilita a identificacao visual dos
# periodos e estados mais criticos sem precisar ler todos os numeros.

tabela = pd.pivot_table(df, index='uf', columns='ano',
                         values='casos_dengue', aggfunc='sum',
                         fill_value=0, margins=True, margins_name='TOTAL')
tabela.columns = [str(c) for c in tabela.columns]

print('Casos de Dengue por Estado e Ano')
display(tabela.style
        .format('{:,.0f}')
        .background_gradient(cmap='Reds',
                              subset=pd.IndexSlice[:, tabela.columns[:-1]])
        .set_caption('Tabela Dinamica - Casos por UF x Ano'))

**Analise:** A tabela dinâmica complementa os gráficos, permitindo consultar
os valores exatos de qualquer estado em qualquer ano. A linha TOTAL no final
mostra o somatório nacional por ano, facilitando a comparação do volume
total entre os diferentes períodos analisados.

## 7. Persistência em SQLite

In [ ]:
# Salvando os dados em um banco SQLite para demonstrar o uso de SQL no projeto
# O SQLite e um banco de dados leve que nao precisa de servidor
# Aqui salvo o dataframe na tabela 'dengue' e depois faço algumas consultas SQL

engine = create_engine('sqlite:///dengue_brasil.sqlite', echo=False)
df.to_sql('dengue', engine, if_exists='replace', index=False)
print('Dados salvos em dengue_brasil.sqlite')

# Realizando consultas SQL para extrair informacoes relevantes
consultas = {
    'Top 5 estados por casos': '''
        SELECT uf, SUM(casos_dengue) AS total
        FROM dengue GROUP BY uf
        ORDER BY total DESC LIMIT 5''',
    'Casos e obitos por ano': '''
        SELECT ano, SUM(casos_dengue) AS casos,
               SUM(obitos) AS obitos,
               ROUND(AVG(incidencia_100k),2) AS incidencia
        FROM dengue GROUP BY ano ORDER BY ano''',
    'Meses mais criticos': '''
        SELECT mes, SUM(casos_dengue) AS total
        FROM dengue GROUP BY mes
        ORDER BY total DESC LIMIT 6'''
}

with engine.connect() as conn:
    for titulo, sql in consultas.items():
        print(f'\n=== {titulo} ===')
        display(pd.read_sql_query(text(sql), conn))

## 8. Interpretação dos Resultados

In [ ]:
# Gerando um resumo automatico com os principais achados da analise
cresc      = df.groupby('ano')['casos_dengue'].sum().pct_change().mean()*100
mes_crit   = MESES[df.groupby('mes')['casos_dengue'].sum().idxmax()-1]
reg_lider  = df.groupby('regiao')['casos_dengue'].sum().idxmax()
r          = df[['chuva_mm','casos_dengue']].corr().iloc[0,1]
forca      = 'forte' if abs(r)>.6 else 'moderada' if abs(r)>.3 else 'fraca'

print('=' * 60)
print('          INTERPRETACAO DOS RESULTADOS')
print('=' * 60)
print(f'''
EVOLUCAO TEMPORAL
   {total_casos:,.0f} casos registrados entre 2015 e 2024.
   Crescimento medio anual: {cresc:+.1f}%.
   Ano de pico: {ano_pico} ({ano_pico_v:,.0f} casos).

SAZONALIDADE
   Mes mais critico: {mes_crit}.
   Periodo de maior risco: janeiro a abril.

DISTRIBUICAO REGIONAL
   Regiao com mais casos: {reg_lider}.
   Estado mais afetado: {uf_top} ({uf_top_v:,.0f} casos).

CORRELACAO CHUVA x DENGUE
   r = {r:.3f} - correlacao {forca}.
   Defasagem tipica de 2 a 4 semanas (ciclo do mosquito).

MUNICIPIO CRITICO
   {mun_top} - {mun_inc:.1f} casos/100k hab.
''')
print('=' * 60)

## 9. Conclusão

Depois de analisar os dados de dengue no Brasil entre 2015 e 2024,
conseguimos responder as perguntas que nos propusemos no início do projeto:

| Pergunta | Resposta encontrada |
|---|---|
| Estados com mais casos? | Norte e Nordeste lideram o ranking |
| Períodos críticos? | Janeiro a abril (verão/outono) |
| Os casos cresceram? | Tendência de alta com anos epidémicos |
| Relação chuva x dengue? | Correlação positiva confirmada |
| Municípios mais afetados? | Ver ranking na seção 6.4 |
| Regiões mais vulneráveis? | Norte e Nordeste |
| Quando agir preventivamente? | Outubro a dezembro |

### Recomendações

Com base nos resultados encontrados, sugiro as seguintes ações:

1. **Campanhas preventivas em outubro e novembro** — antes do pico de casos
2. **Reforço hospitalar entre janeiro e abril** — período crítico
3. **Atenção especial aos municípios com maior incidência** — focos prioritários
4. **Monitoramento climatico integrado** — alertas com base nas previsões de chuva
5. **Políticas regionais diferenciadas** — Norte e Nordeste precisam de mais atenção

---
> *Projeto G2 — Dados simulados para fins didaticos.*